## 2021.acl-long.353

In [ ]:
import csv
from collections import defaultdict

# Allowed labels based on your CSV
ALLOWED_LABELS = {'Method', 'Task', 'Dataset', 'Metric'}

# ----------------------------------------------------------------------
# Load ground truth from CSV (single document expected)
# ----------------------------------------------------------------------
def load_ground_truth(csv_path, target_doc_key=None):
    """
    Returns a dict mapping doc_key -> list of entities.
    If target_doc_key is given, only that document is kept.
    """
    true_by_doc = defaultdict(list)
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            label = row['ner_label']
            if label in ALLOWED_LABELS:
                doc_key = row['doc_key']
                if target_doc_key is None or doc_key == target_doc_key:
                    true_by_doc[doc_key].append({
                        'start': int(row['start_token']),
                        'end': int(row['end_token']),
                        'label': label,
                        'text': row.get('text', '')
                    })
    return true_by_doc

def load_predictions(csv_path, target_doc_key=None):
    """Same as above for predictions."""
    pred_by_doc = defaultdict(list)
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            label = row['ner_label']
            if label in ALLOWED_LABELS:
                doc_key = row['doc_key']
                if target_doc_key is None or doc_key == target_doc_key:
                    pred_by_doc[doc_key].append({
                        'start': int(row['start_token']),
                        'end': int(row['end_token']),
                        'label': label,
                        'text': row.get('text', '')
                    })
    return pred_by_doc

# ----------------------------------------------------------------------
# Exact-match evaluation (no partial overlap)
# ----------------------------------------------------------------------
def evaluate_exact(true_entities, pred_entities):
    """Compute exact-match precision, recall, F1."""
    # Convert to sets of (start, end, label)
    true_set = {(e['start'], e['end'], e['label']) for e in true_entities}
    pred_set = {(e['start'], e['end'], e['label']) for e in pred_entities}

    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    return {'precision': prec, 'recall': rec, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}

def per_label_exact(true_entities, pred_entities):
    """Return exact metrics for each label."""
    # Group by label
    true_by_label = defaultdict(list)
    pred_by_label = defaultdict(list)
    for e in true_entities:
        true_by_label[e['label']].append((e['start'], e['end']))
    for e in pred_entities:
        pred_by_label[e['label']].append((e['start'], e['end']))

    label_stats = {}
    for label in ALLOWED_LABELS:
        true_spans = set(true_by_label.get(label, []))
        pred_spans = set(pred_by_label.get(label, []))
        tp = len(true_spans & pred_spans)
        fp = len(pred_spans - true_spans)
        fn = len(true_spans - pred_spans)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        label_stats[label] = {'precision': prec, 'recall': rec, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}
    return label_stats

# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
def main():
    # === CONFIGURATION ===
    GROUND_TRUTH_CSV = '/content/SciNLP_Entities_Test_Set.csv'   # change this
    PREDICTION_CSV   = '/content/predictions_csv_hard/2021.acl-long.353.csv'    # change this
    # Optional: if CSVs contain multiple documents, specify which doc_key to use.
    # Leave as None to use the first document found in ground truth.
    TARGET_DOC_KEY = None
    # =====================

    # Load data
    true_by_doc = load_ground_truth(GROUND_TRUTH_CSV, TARGET_DOC_KEY)
    if not true_by_doc:
        print("No ground truth entities found. Check ALLOWED_LABELS and file content.")
        return

    # If no specific doc_key given, take the first one
    if TARGET_DOC_KEY is None:
        doc_key = list(true_by_doc.keys())[0]
        print(f"No target doc_key specified. Using first document: {doc_key}")
    else:
        doc_key = TARGET_DOC_KEY
        if doc_key not in true_by_doc:
            print(f"Document '{doc_key}' not found in ground truth.")
            return

    true_entities = true_by_doc[doc_key]

    # Load predictions for the same document
    pred_by_doc = load_predictions(PREDICTION_CSV, doc_key)
    pred_entities = pred_by_doc.get(doc_key, [])
    print(f"Document: {doc_key}")
    print(f"Ground truth entities: {len(true_entities)}")
    print(f"Prediction entities:   {len(pred_entities)}")

    # Overall exact evaluation (micro across all labels)
    overall = evaluate_exact(true_entities, pred_entities)
    print("\n=== OVERALL EXACT MATCH ===")
    print(f"Precision: {overall['precision']:.4f}")
    print(f"Recall:    {overall['recall']:.4f}")
    print(f"F1:        {overall['f1']:.4f}")
    print(f"TP: {overall['tp']}, FP: {overall['fp']}, FN: {overall['fn']}")

    # Per-label exact evaluation
    label_stats = per_label_exact(true_entities, pred_entities)
    print("\n=== PER-LABEL EXACT MATCH ===")
    for label in ALLOWED_LABELS:
        s = label_stats[label]
        print(f"{label:12s} P={s['precision']:.3f} R={s['recall']:.3f} F1={s['f1']:.3f}  (TP={s['tp']}, FP={s['fp']}, FN={s['fn']})")

    # Macro-average F1 over labels
    macro_f1 = sum(s['f1'] for s in label_stats.values()) / len(ALLOWED_LABELS)
    print(f"\nMacro-average F1 (exact): {macro_f1:.4f}")

if __name__ == '__main__':
    main()

No target doc_key specified. Using first document: 2021.acl-long.353
Document: 2021.acl-long.353
Ground truth entities: 401
Prediction entities:   338

=== OVERALL EXACT MATCH ===
Precision: 0.7988
Recall:    0.6733
F1:        0.7307
TP: 270, FP: 68, FN: 131

=== PER-LABEL EXACT MATCH ===
Dataset      P=0.788 R=0.634 F1=0.703  (TP=26, FP=7, FN=15)
Task         P=0.663 R=0.750 F1=0.704  (TP=57, FP=29, FN=19)
Method       P=0.867 R=0.642 F1=0.738  (TP=163, FP=25, FN=91)
Metric       P=0.774 R=0.800 F1=0.787  (TP=24, FP=7, FN=6)

Macro-average F1 (exact): 0.7327


# P17-1058

In [ ]:
import csv
from collections import defaultdict

# Allowed labels based on your CSV
ALLOWED_LABELS = {'Method', 'Task', 'Dataset', 'Metric'}

# ----------------------------------------------------------------------
# Load ground truth from CSV (single document expected)
# ----------------------------------------------------------------------
def load_ground_truth(csv_path, target_doc_key=None):
    """
    Returns a dict mapping doc_key -> list of entities.
    If target_doc_key is given, only that document is kept.
    """
    true_by_doc = defaultdict(list)
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            label = row['ner_label']
            if label in ALLOWED_LABELS:
                doc_key = row['doc_key']
                if target_doc_key is None or doc_key == target_doc_key:
                    true_by_doc[doc_key].append({
                        'start': int(row['start_token']),
                        'end': int(row['end_token']),
                        'label': label,
                        'text': row.get('text', '')
                    })
    return true_by_doc

def load_predictions(csv_path, target_doc_key=None):
    """Same as above for predictions."""
    pred_by_doc = defaultdict(list)
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            label = row['ner_label']
            if label in ALLOWED_LABELS:
                doc_key = row['doc_key']
                if target_doc_key is None or doc_key == target_doc_key:
                    pred_by_doc[doc_key].append({
                        'start': int(row['start_token']),
                        'end': int(row['end_token']),
                        'label': label,
                        'text': row.get('text', '')
                    })
    return pred_by_doc

# ----------------------------------------------------------------------
# Exact-match evaluation (no partial overlap)
# ----------------------------------------------------------------------
def evaluate_exact(true_entities, pred_entities):
    """Compute exact-match precision, recall, F1."""
    # Convert to sets of (start, end, label)
    true_set = {(e['start'], e['end'], e['label']) for e in true_entities}
    pred_set = {(e['start'], e['end'], e['label']) for e in pred_entities}

    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    return {'precision': prec, 'recall': rec, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}

def per_label_exact(true_entities, pred_entities):
    """Return exact metrics for each label."""
    # Group by label
    true_by_label = defaultdict(list)
    pred_by_label = defaultdict(list)
    for e in true_entities:
        true_by_label[e['label']].append((e['start'], e['end']))
    for e in pred_entities:
        pred_by_label[e['label']].append((e['start'], e['end']))

    label_stats = {}
    for label in ALLOWED_LABELS:
        true_spans = set(true_by_label.get(label, []))
        pred_spans = set(pred_by_label.get(label, []))
        tp = len(true_spans & pred_spans)
        fp = len(pred_spans - true_spans)
        fn = len(true_spans - pred_spans)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        label_stats[label] = {'precision': prec, 'recall': rec, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}
    return label_stats

# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
def main():
    # === CONFIGURATION ===
    GROUND_TRUTH_CSV = '/content/SciNLP_Entities_Test_Set.csv'   # change this
    PREDICTION_CSV   = '/content/predictions_csv_hard/P17-1058.grobid.csv'    # change this
    # Optional: if CSVs contain multiple documents, specify which doc_key to use.
    # Leave as None to use the first document found in ground truth.
    TARGET_DOC_KEY = 'P17-1058.grobid'
    # =====================

    # Load data
    true_by_doc = load_ground_truth(GROUND_TRUTH_CSV, TARGET_DOC_KEY)
    if not true_by_doc:
        print("No ground truth entities found. Check ALLOWED_LABELS and file content.")
        return

    # If no specific doc_key given, take the first one
    if TARGET_DOC_KEY is None:
        doc_key = list(true_by_doc.keys())[0]
        print(f"No target doc_key specified. Using first document: {doc_key}")
    else:
        doc_key = TARGET_DOC_KEY
        if doc_key not in true_by_doc:
            print(f"Document '{doc_key}' not found in ground truth.")
            return

    true_entities = true_by_doc[doc_key]

    # Load predictions for the same document
    pred_by_doc = load_predictions(PREDICTION_CSV, doc_key)
    pred_entities = pred_by_doc.get(doc_key, [])
    print(f"Document: {doc_key}")
    print(f"Ground truth entities: {len(true_entities)}")
    print(f"Prediction entities:   {len(pred_entities)}")

    # Overall exact evaluation (micro across all labels)
    overall = evaluate_exact(true_entities, pred_entities)
    print("\n=== OVERALL EXACT MATCH ===")
    print(f"Precision: {overall['precision']:.4f}")
    print(f"Recall:    {overall['recall']:.4f}")
    print(f"F1:        {overall['f1']:.4f}")
    print(f"TP: {overall['tp']}, FP: {overall['fp']}, FN: {overall['fn']}")

    # Per-label exact evaluation
    label_stats = per_label_exact(true_entities, pred_entities)
    print("\n=== PER-LABEL EXACT MATCH ===")
    for label in ALLOWED_LABELS:
        s = label_stats[label]
        print(f"{label:12s} P={s['precision']:.3f} R={s['recall']:.3f} F1={s['f1']:.3f}  (TP={s['tp']}, FP={s['fp']}, FN={s['fn']})")

    # Macro-average F1 over labels
    macro_f1 = sum(s['f1'] for s in label_stats.values()) / len(ALLOWED_LABELS)
    print(f"\nMacro-average F1 (exact): {macro_f1:.4f}")

if __name__ == '__main__':
    main()

Document: P17-1058.grobid
Ground truth entities: 42
Prediction entities:   191

=== OVERALL EXACT MATCH ===
Precision: 0.1675
Recall:    0.7619
F1:        0.2747
TP: 32, FP: 159, FN: 10

=== PER-LABEL EXACT MATCH ===
Dataset      P=0.629 R=1.000 F1=0.772  (TP=22, FP=13, FN=0)
Task         P=0.000 R=0.000 F1=0.000  (TP=0, FP=3, FN=2)
Method       P=0.075 R=0.556 F1=0.132  (TP=10, FP=123, FN=8)
Metric       P=0.000 R=0.000 F1=0.000  (TP=0, FP=20, FN=0)

Macro-average F1 (exact): 0.2261


# 2024.acl-long.108-1-9.grobid

In [ ]:
import csv
from collections import defaultdict

# Allowed labels based on your CSV
ALLOWED_LABELS = {'Method', 'Task', 'Dataset', 'Metric'}

# ----------------------------------------------------------------------
# Load ground truth from CSV (single document expected)
# ----------------------------------------------------------------------
def load_ground_truth(csv_path, target_doc_key=None):
    """
    Returns a dict mapping doc_key -> list of entities.
    If target_doc_key is given, only that document is kept.
    """
    true_by_doc = defaultdict(list)
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            label = row['ner_label']
            if label in ALLOWED_LABELS:
                doc_key = row['doc_key']
                if target_doc_key is None or doc_key == target_doc_key:
                    true_by_doc[doc_key].append({
                        'start': int(row['start_token']),
                        'end': int(row['end_token']),
                        'label': label,
                        'text': row.get('text', '')
                    })
    return true_by_doc

def load_predictions(csv_path, target_doc_key=None):
    """Same as above for predictions."""
    pred_by_doc = defaultdict(list)
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            label = row['ner_label']
            if label in ALLOWED_LABELS:
                doc_key = row['doc_key']
                if target_doc_key is None or doc_key == target_doc_key:
                    pred_by_doc[doc_key].append({
                        'start': int(row['start_token']),
                        'end': int(row['end_token']),
                        'label': label,
                        'text': row.get('text', '')
                    })
    return pred_by_doc

# ----------------------------------------------------------------------
# Exact-match evaluation (no partial overlap)
# ----------------------------------------------------------------------
def evaluate_exact(true_entities, pred_entities):
    """Compute exact-match precision, recall, F1."""
    # Convert to sets of (start, end, label)
    true_set = {(e['start'], e['end'], e['label']) for e in true_entities}
    pred_set = {(e['start'], e['end'], e['label']) for e in pred_entities}

    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    return {'precision': prec, 'recall': rec, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}

def per_label_exact(true_entities, pred_entities):
    """Return exact metrics for each label."""
    # Group by label
    true_by_label = defaultdict(list)
    pred_by_label = defaultdict(list)
    for e in true_entities:
        true_by_label[e['label']].append((e['start'], e['end']))
    for e in pred_entities:
        pred_by_label[e['label']].append((e['start'], e['end']))

    label_stats = {}
    for label in ALLOWED_LABELS:
        true_spans = set(true_by_label.get(label, []))
        pred_spans = set(pred_by_label.get(label, []))
        tp = len(true_spans & pred_spans)
        fp = len(pred_spans - true_spans)
        fn = len(true_spans - pred_spans)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        label_stats[label] = {'precision': prec, 'recall': rec, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}
    return label_stats

# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
def main():
    # === CONFIGURATION ===
    GROUND_TRUTH_CSV = '/content/SciNLP_Entities_Test_Set.csv'   # change this
    PREDICTION_CSV   = '/content/predictions_csv_hard/2024.acl-long.108-1-9.grobid.csv'    # change this
    # Optional: if CSVs contain multiple documents, specify which doc_key to use.
    # Leave as None to use the first document found in ground truth.
    TARGET_DOC_KEY = '2024.acl-long.108-1-9.grobid'
    # =====================

    # Load data
    true_by_doc = load_ground_truth(GROUND_TRUTH_CSV, TARGET_DOC_KEY)
    if not true_by_doc:
        print("No ground truth entities found. Check ALLOWED_LABELS and file content.")
        return

    # If no specific doc_key given, take the first one
    if TARGET_DOC_KEY is None:
        doc_key = list(true_by_doc.keys())[0]
        print(f"No target doc_key specified. Using first document: {doc_key}")
    else:
        doc_key = TARGET_DOC_KEY
        if doc_key not in true_by_doc:
            print(f"Document '{doc_key}' not found in ground truth.")
            return

    true_entities = true_by_doc[doc_key]

    # Load predictions for the same document
    pred_by_doc = load_predictions(PREDICTION_CSV, doc_key)
    pred_entities = pred_by_doc.get(doc_key, [])
    print(f"Document: {doc_key}")
    print(f"Ground truth entities: {len(true_entities)}")
    print(f"Prediction entities:   {len(pred_entities)}")

    # Overall exact evaluation (micro across all labels)
    overall = evaluate_exact(true_entities, pred_entities)
    print("\n=== OVERALL EXACT MATCH ===")
    print(f"Precision: {overall['precision']:.4f}")
    print(f"Recall:    {overall['recall']:.4f}")
    print(f"F1:        {overall['f1']:.4f}")
    print(f"TP: {overall['tp']}, FP: {overall['fp']}, FN: {overall['fn']}")

    # Per-label exact evaluation
    label_stats = per_label_exact(true_entities, pred_entities)
    print("\n=== PER-LABEL EXACT MATCH ===")
    for label in ALLOWED_LABELS:
        s = label_stats[label]
        print(f"{label:12s} P={s['precision']:.3f} R={s['recall']:.3f} F1={s['f1']:.3f}  (TP={s['tp']}, FP={s['fp']}, FN={s['fn']})")

    # Macro-average F1 over labels
    macro_f1 = sum(s['f1'] for s in label_stats.values()) / len(ALLOWED_LABELS)
    print(f"\nMacro-average F1 (exact): {macro_f1:.4f}")

if __name__ == '__main__':
    main()

Document: 2024.acl-long.108-1-9.grobid
Ground truth entities: 302
Prediction entities:   419

=== OVERALL EXACT MATCH ===
Precision: 0.6181
Recall:    0.8576
F1:        0.7184
TP: 259, FP: 160, FN: 43

=== PER-LABEL EXACT MATCH ===
Dataset      P=0.765 R=0.812 F1=0.788  (TP=13, FP=4, FN=3)
Task         P=0.670 R=0.843 F1=0.747  (TP=59, FP=29, FN=11)
Method       P=0.580 R=0.868 F1=0.695  (TP=171, FP=124, FN=26)
Metric       P=0.842 R=0.842 F1=0.842  (TP=16, FP=3, FN=3)

Macro-average F1 (exact): 0.7680
